# CFPB Narrative Coverage

**Question:** how many complaints in `data/raw/paydayComplaints.csv` have a publicly
available consumer narrative in the CFPB Consumer Complaint Database Narratives Archive?

**Scope:** measurement only. This notebook does **not** build a scoring model, does not
run any NLP, and does not touch the frontend or backend. The raw CSV is read-only here.

**Method:** download the official FOIA *CCDB Export* ZIPs from the
[Narratives Archive](https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/),
then join to our extract on `Complaint ID`.

**Reproducibility:** the first run downloads ~950 MB into `data/raw/narratives_cache/`
(git-ignored) and takes roughly 5 minutes. Later runs reuse the cached join and finish
in seconds. To force a clean rebuild, delete that directory.

In [1]:
import os
import subprocess
import time
import zipfile

import pandas as pd

REPO = "/Users/mihirkale/cdc"
RAW = f"{REPO}/data/raw/paydayComplaints.csv"
CACHE = f"{REPO}/data/raw/narratives_cache"
BASE = "https://files.consumerfinance.gov/f/documents/"

# Sequential, non-overlapping FOIA CCDB exports. Export 4 is required because our
# extract starts 2023-08-25, inside its Nov 2022 - Aug 2023 window.
EXPORTS = [
    (4,  "CCDB_Export_4_November_2022_through_August_2023.zip"),
    (5,  "CCDB_Export_5_September_2023_through_March_2024.zip"),
    (6,  "CCDB_Export_6_April_2024_through_July_2024.zip"),
    (7,  "CCDB_Export_7_August_2024_through_October_2024.zip"),
    (8,  "CCDB_Export_8_November_2024_through_December_2024.zip"),
    (9,  "CCDB_Export_9_January_2025_through_February_2025.zip"),
    (10, "CCDB_Export_10_March_2025_through_April_2025.zip"),
    (11, "CCDB_Export_11_May_2025_through_June_2025.zip"),
    (12, "CCDB_Export_12_July_2025_through_August_2025.zip"),
    (13, "CCDB_Export_13_September_2025_through_October_2025.zip"),
    (14, "CCDB_Export_14_November_2025_through_December_2025.zip"),
    (15, "CCDB_Export_15_January_2026_through_February_2026.zip"),
    (16, "CCDB_Export_16_March_2026.zip"),
    (17, "CCDB_Export_17_April_2026.zip"),
    (18, "CCDB_Export_18_May_2026.zip"),
    (19, "CCDB_Export_19_June_2026.zip"),
    (20, "CCDB_Export_20_July_2026.zip"),
    (21, "CCDB_Export_21_August_2026.zip"),
]

USECOLS = [
    "Complaint ID",
    "Date received",
    "Sub-product",
    "Company",
    "Consumer complaint narrative",
]

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1. Schema of our existing extract

In [2]:
ours = pd.read_csv(RAW)
print(f"rows={len(ours):,}  cols={ours.shape[1]}  dup_ids={ours['Complaint ID'].duplicated().sum()}")
print(f"Date received: {ours['Date received'].min()}  ->  {ours['Date received'].max()}")
print(f"companies={ours['Company'].nunique()}   sub-products={ours['Sub-product'].nunique()}")
ours.dtypes.to_frame("dtype").T

rows=38,375  cols=15  dup_ids=0
Date received: 2023-08-25T00:45:25.000Z  ->  2026-09-25T17:44:12.000Z
companies=1162   sub-products=8


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
dtype,str,str,str,str,str,str,str,str,str,str,str,str,str,str,int64


In [3]:
ours.head(3).T

,0,1,2
Date received,2023-08-25T04:08:45.000Z,2023-08-25T00:45:25.000Z,2023-08-25T12:16:47.000Z
Product,"Payday loan, title loan, personal loan, or advance loan","Payday loan, title loan, personal loan, or advance loan","Payday loan, title loan, personal loan, or advance loan"
Sub-product,Personal line of credit,Payday loan,Installment loan
Issue,Getting a line of credit,Charged fees or interest you didn't expect,Problem when making payments
Sub-issue,NaN,NaN,NaN
Company public response,NaN,NaN,NaN
Company,TD BANK US HOLDING COMPANY,"ENOVA INTERNATIONAL, INC.","Uetsa Tsakits, Inc. d/b/a MaxLend"
State,NY,VA,AZ
ZIP code,10001,23224,85138
Tags,NaN,NaN,Servicemember


## 2. The archive

The FOIA archive page publishes one section, *Full Records*: 21 sequential,
non-overlapping `CCDB_Export_N.zip` files. Each member CSV carries the same 16
columns as the live CCDB — which is our 15 columns **plus**
`Consumer complaint narrative`. `Sub-product` is confirmed byte-identical in name,
so `Complaint ID` is a clean join key.

Note the archive's own stated ceiling: it covers complaints received through
**2026-08-14**, while our extract runs to 2026-09-25.

In [4]:
def _peek(fn):
    path = os.path.join(CACHE, fn)
    if not os.path.exists(path):
        return None
    with zipfile.ZipFile(path) as zf:
        with zf.open(zf.namelist()[0]) as f:
            header = f.readline().decode("utf-8", "replace").strip()
    return header.strip('"').split(",")


_peek(EXPORTS[0][1]) and None
latest = [f for n, f in EXPORTS if n == 21][0]
arch_cols = _peek(latest)
if arch_cols is None:
    print("archive not cached yet; run the next cell first, then re-run this one.")
else:
    print(f"archive columns ({len(arch_cols)}):")
    for i, c in enumerate(arch_cols):
        mark = "  <-- EXTRA (not in our extract)" if c not in ours.columns else ""
        print(f"  {i:>2}  {c}{mark}")
    print()
    print("in our extract but not in archive:", [c for c in ours.columns if c not in arch_cols])

archive columns (16):
   0  Date received
   1  Product
   2  Sub-product
   3  Issue
   4  Sub-issue
   5  Consumer complaint narrative  <-- EXTRA (not in our extract)
   6  Company public response
   7  Company
   8  State
   9  ZIP code
  10  Tags
  11  Submitted via
  12  Date sent to company
  13  Company response to consumer
  14  Timely response?
  15  Complaint ID

in our extract but not in archive: []


## 3. Download and join on `Complaint ID`

In [5]:
def _fetch(fn):
    """files.consumerfinance.gov 403s urllib's default UA; curl works."""
    path = os.path.join(CACHE, fn)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path, True
    os.makedirs(CACHE, exist_ok=True)
    tmp = path + ".part"
    for attempt in range(3):
        r = subprocess.run(
            ["curl", "-sSL", "--fail", "--retry", "3", "-o", tmp, BASE + fn],
            capture_output=True,
        )
        if r.returncode == 0 and os.path.exists(tmp) and os.path.getsize(tmp) > 0:
            os.replace(tmp, path)
            return path, False
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"download failed: {fn}")


def build_cache():
    """Stream every export, keep only rows whose Complaint ID is in our extract."""
    ours_ids = pd.read_csv(RAW, usecols=["Complaint ID"])["Complaint ID"].to_numpy()
    stats, parts = [], []
    for n, fn in EXPORTS:
        t0 = time.time()
        path, cached = _fetch(fn)
        tot = narr = hit = 0
        chunks = []
        with zipfile.ZipFile(path) as zf, zf.open(zf.namelist()[0]) as fh:
            for ch in pd.read_csv(fh, usecols=USECOLS, chunksize=250_000, low_memory=False):
                tot += len(ch)
                s = ch["Consumer complaint narrative"].fillna("").str.strip()
                narr += int(s.ne("").sum())
                m = ch["Complaint ID"].isin(ours_ids)
                if m.any():
                    sub = ch.loc[m].copy()
                    sub["Consumer complaint narrative"] = s.loc[m].to_numpy()
                    chunks.append(sub)
                    hit += int(m.sum())
        if chunks:
            got = pd.concat(chunks, ignore_index=True)
            got.insert(0, "export", n)
            parts.append(got)
        stats.append({
            "export": n, "file": fn,
            "archive_rows": tot, "archive_narratives": narr,
            "archive_narrative_pct": round(narr / tot * 100, 3) if tot else None,
            "our_ids_found": hit, "cached": cached,
        })
        print(f"export {n:>2}: archive={tot:>9,}  narratives={narr:>9,} "
              f"({(narr / tot * 100 if tot else 0):5.2f}%)  our_ids={hit:>6,}  "
              f"({time.time() - t0:.0f}s{' cached' if cached else ''})", flush=True)
    pd.DataFrame(stats).to_csv(f"{CACHE}/export_stats.csv", index=False)
    out = pd.concat(parts, ignore_index=True)
    out = out.drop_duplicates(subset=["Complaint ID"], keep="first")
    out.to_csv(f"{CACHE}/matched_narratives.csv", index=False)
    return out


MATCHED_PATH = f"{CACHE}/matched_narratives.csv"

if os.path.exists(MATCHED_PATH):
    print("using cached join ->", MATCHED_PATH)
    matched = pd.read_csv(MATCHED_PATH)
    export_stats = pd.read_csv(f"{CACHE}/export_stats.csv")
else:
    print("no cache found; downloading ~950 MB (about 5 minutes)\n")
    matched = build_cache()
    export_stats = pd.read_csv(f"{CACHE}/export_stats.csv")

print(f"\nmatched rows={len(matched):,}  unique ids={matched['Complaint ID'].nunique():,}")

using cached join -> /Users/mihirkale/cdc/data/raw/narratives_cache/matched_narratives.csv



matched rows=37,322  unique ids=37,322


### 3a. Per-export audit trail

In [6]:
export_stats

,export,file,archive_rows,archive_narratives,archive_narrative_pct,our_ids_found,our_ids_found_pct,zip_mb,cached
0,4,CCDB_Export_4_November_2022_through_August_2023.zip,987495,388495,39.3415,117,0.3049,110.2,False
1,5,CCDB_Export_5_September_2023_through_March_2024.zip,945532,318804,33.7169,4777,12.4482,96.2,True
2,6,CCDB_Export_6_April_2024_through_July_2024.zip,839903,276332,32.9005,2723,7.0958,73.6,False
3,7,CCDB_Export_7_August_2024_through_October_2024.zip,824514,229944,27.8884,2369,6.1733,68.1,False
4,8,CCDB_Export_8_November_2024_through_December_2024.zip,591058,154916,26.2099,1609,4.1928,50.1,False
5,9,CCDB_Export_9_January_2025_through_February_2025.zip,811019,265405,32.7249,1848,4.8156,72.0,False
6,10,CCDB_Export_10_March_2025_through_April_2025.zip,803073,210147,26.1679,1754,4.5707,65.7,False
7,11,CCDB_Export_11_May_2025_through_June_2025.zip,827343,219632,26.5467,2054,5.3524,71.1,False
8,12,CCDB_Export_12_July_2025_through_August_2025.zip,968761,234404,24.1963,2377,6.1941,80.3,False
9,13,CCDB_Export_13_September_2025_through_October_2025.zip,1023636,157821,15.4177,2423,6.3140,64.2,False


In [7]:
# Narrative supply is collapsing over calendar time, in the archive as a whole
# (not just in our slice).
tot = export_stats["archive_rows"].sum()
nar = export_stats["archive_narratives"].sum()
print(f"archive-wide narratives: {nar:,} / {tot:,} = {nar / tot * 100:.2f}%")
print("\nlast 8 exports (oldest -> newest):")
print(export_stats.tail(8)[["export", "archive_rows", "archive_narratives", "archive_narrative_pct"]].to_string(index=False))

archive-wide narratives: 2,711,935 / 14,482,997 = 18.72%

last 8 exports (oldest -> newest):
 export  archive_rows  archive_narratives  archive_narrative_pct
     14       1009132              134641                13.3423
     15       1038383               34351                 3.3081
     16        614945               21640                 3.5190
     17        618293               21052                 3.4049
     18        619940               17528                 2.8274
     19        620794               16820                 2.7094
     20        672036               10002                 1.4883
     21        667140                   1                 0.0001


### 3b. Join validation

Does the archive agree with our extract on the rows it matched?

In [8]:
matched["has_narrative"] = (
    matched["Consumer complaint narrative"].fillna("").str.strip().ne("")
)
print(f"matched ids           : {len(matched):,}")
print(f"  with narrative      : {int(matched['has_narrative'].sum()):,}")
print(f"  blank/absent        : {int((~matched['has_narrative']).sum()):,}")

o = ours.set_index("Complaint ID")
a = matched.set_index("Complaint ID")
idx = a.index
for col in ["Sub-product", "Company"]:
    bad = int((a.loc[idx, col].astype(str) != o.loc[idx, col].astype(str)).sum())
    print(f"field disagreement on {col:<12}: {bad}")

miss = ours[~ours["Complaint ID"].isin(matched["Complaint ID"])]
print(f"\nunmatched ids: {len(miss):,} ({len(miss) / len(ours) * 100:.2f}%)")
print("unmatched by month (last 8):")
print(miss["Date received"].str[:7].value_counts().sort_index().tail(8).to_string())

matched ids           : 37,322
  with narrative      : 18,863
  blank/absent        : 18,459


field disagreement on Sub-product : 0


field disagreement on Company     : 284

unmatched ids: 1,053 (2.74%)
unmatched by month (last 8):
Date received
2026-03      5
2026-04      1
2026-05      2
2026-06      8
2026-07     29
2026-08     72
2026-09    936


In [9]:
# The handful of Company disagreements are worth naming rather than averaging away.
o = ours.set_index("Complaint ID")
a = matched.set_index("Complaint ID")
idx = a.index
mis = pd.DataFrame({
    "our_extract": o.loc[idx, "Company"],
    "archive": a.loc[idx, "Company"],
    "Date received": o.loc[idx, "Date received"],
})
mis = mis[mis["our_extract"].astype(str) != mis["archive"].astype(str)]
print(f"rows with a different Company name: {len(mis):,}  "
      f"across {mis['our_extract'].nunique()} distinct names\n")
print(mis.groupby(["our_extract", "archive"]).size().sort_values(ascending=False).to_string())

rows with a different Company name: 284  across 3 distinct names

our_extract                     archive                     
Ningo Lending LLC dba Spotloan  BlueChip Financial              254
STATE EMPLOYEES’ CREDIT UNION   STATE EMPLOYEES CREDIT UNION     29
American First Funding LLC      LAKEVIEW LENDING LLC              1


## 4. Coverage

Building the analysis frame: one row per complaint in our extract, with a
`has_narrative` flag.

In [10]:
df = ours.merge(
    matched[["Complaint ID", "has_narrative", "Consumer complaint narrative"]],
    on="Complaint ID",
    how="left",
)
df["has_narrative"] = df["has_narrative"].fillna(False).astype(bool)
df["narrative"] = df["Consumer complaint narrative"].fillna("").str.strip()
df["year"] = df["Date received"].str[:4]
df["month"] = df["Date received"].str[:7]

n = len(df)
k = int(df["has_narrative"].sum())
print(f"total complaints in our dataset : {n:,}")
print(f"with a narrative                : {k:,}  ({k / n * 100:.2f}%)")
print(f"without                         : {n - k:,}  ({(n - k) / n * 100:.2f}%)")
print(f"unique companies with >=1 narrative: {df.loc[df.has_narrative, 'Company'].nunique():,} "
      f"of {df['Company'].nunique():,}")

total complaints in our dataset : 38,375
with a narrative                : 18,863  (49.15%)
without                         : 19,512  (50.85%)
unique companies with >=1 narrative: 838 of 1,162


### 4a. Coverage by `Sub-product`

In [11]:
sp = df.groupby("Sub-product")["has_narrative"].agg(total="size", with_narrative="sum")
sp["pct"] = (sp["with_narrative"] / sp["total"] * 100).round(2)
sp["companies"] = df.groupby("Sub-product")["Company"].nunique()
sp["cos_with_narr"] = (
    df[df.has_narrative].groupby("Sub-product")["Company"].nunique()
)
sp["cos_with_narr_pct"] = (sp["cos_with_narr"] / sp["companies"] * 100).round(1)
sp[["total", "with_narrative", "pct", "companies", "cos_with_narr", "cos_with_narr_pct"]] \
  .sort_values("total", ascending=False)

,total,with_narrative,pct,companies,cos_with_narr,cos_with_narr_pct
Sub-product,,,,,,
Installment loan,21835,10127,46.38,805,568,70.6
Personal line of credit,6712,3349,49.90,508,358,70.5
Payday loan,6024,3272,54.32,483,354,73.3
Title loan,2488,1432,57.56,211,158,74.9
Other advances of future income,731,351,48.02,235,146,62.1
Earned wage access,396,224,56.57,93,61,65.6
Pawn loan,127,74,58.27,53,33,62.3
Tax refund anticipation loan or check,62,34,54.84,31,20,64.5


### 4b. Payday loans specifically

In [12]:
PAYDAY = "Payday loan"
p_total = int((df["Sub-product"] == PAYDAY).sum())
p_narr = int(df.loc[df["Sub-product"] == PAYDAY, "has_narrative"].sum())
print(f"Sub-product == {PAYDAY!r}")
print(f"  complaints            : {p_total:,}")
print(f"  with narrative        : {p_narr:,}  ({p_narr / p_total * 100:.2f}%)")
print(f"  companies (all)       : {df.loc[df['Sub-product'] == PAYDAY, 'Company'].nunique():,}")
print(f"  companies with >=1    : {df.loc[(df['Sub-product'] == PAYDAY) & df.has_narrative, 'Company'].nunique():,}")

Sub-product == 'Payday loan'
  complaints            : 6,024
  with narrative        : 3,272  (54.32%)
  companies (all)       : 483
  companies with >=1    : 354


### 4c. Coverage by year

In [13]:
yr = df.groupby("year")["has_narrative"].agg(total="size", with_narrative="sum")
yr["pct"] = (yr["with_narrative"] / yr["total"] * 100).round(2)
yr

,total,with_narrative,pct
year,,,
2023,2559,1492,58.30
2024,9036,5211,57.67
2025,13137,7805,59.41
2026,13643,4355,31.92


In [14]:
# The 2026 figure is not a smooth decline, it is a cliff. Monthly detail:
mo = df.groupby("month")["has_narrative"].agg(total="size", with_narrative="sum")
mo["pct"] = (mo["with_narrative"] / mo["total"] * 100).round(1)
print("last 20 months:")
print(mo.tail(20).to_string())
print("\nmonths with 0% narrative coverage:",
      ", ".join(mo.index[mo["with_narrative"] == 0].tolist()) or "none")

last 20 months:
         total  with_narrative   pct
month                               
2025-02    750             453  60.4
2025-03    862             525  60.9
2025-04    892             555  62.2
2025-05    985             624  63.4
2025-06   1069             629  58.8
2025-07   1244             817  65.7
2025-08   1133             721  63.6
2025-09   1242             749  60.3
2025-10   1181             654  55.4
2025-11   1198             750  62.6
2025-12   1483             660  44.5
2026-01   1689             800  47.4
2026-02   1311             405  30.9
2026-03   1505             673  44.7
2026-04   1532             776  50.7
2026-05   1533             656  42.8
2026-06   1596             636  39.8
2026-07   1818             409  22.5
2026-08   1723               0   0.0
2026-09    936               0   0.0

months with 0% narrative coverage: 2026-08, 2026-09


## 5. Payday-loan narrative counts per company

In [15]:
pay = df[(df["Sub-product"] == PAYDAY) & df["has_narrative"]]
per_company = pay.groupby("Company").size().sort_values(ascending=False)

print(f"payday companies with >=1 narrative : {len(per_company):,}")
for k_ in (5, 10, 20, 50):
    print(f"payday companies with >={k_:<3}       : {int((per_company >= k_).sum()):,}")
print()
print(f"median narratives per payday company : {per_company.median():.0f}")
print(f"mean                                : {per_company.mean():.2f}")
print(f"max                                 : {per_company.max()}")
print(f"total payday narratives             : {per_company.sum():,}")
print(f"share of all narratives from top 16 : "
      f"{per_company.head(16).sum() / per_company.sum() * 100:.1f}%")

payday companies with >=1 narrative : 354
payday companies with >=5         : 114
payday companies with >=10        : 76
payday companies with >=20        : 42
payday companies with >=50        : 16

median narratives per payday company : 2
mean                                : 9.24
max                                 : 197
total payday narratives             : 3,272
share of all narratives from top 16 : 42.8%


In [16]:
per_company.head(20).to_frame("narratives")

,narratives
Company,
CCF Intermediate Holdings LLC,197
"ENOVA INTERNATIONAL, INC.",141
"Dave Operating, LLC",127
WLCC,106
Rosebud Economic Development Corporation,103
Ningo Lending LLC dba Spotloan,85
"Uprova Credit, LLC",78
MoneyLion Inc.,77
Wolf River Development Company,76


In [17]:
# How much of the payday long tail is simply too thin to model?
buckets = pd.cut(per_company, [0, 1, 4, 9, 19, 49, 10**6],
                 labels=["1", "2-4", "5-9", "10-19", "20-49", "50+"])
dist = buckets.value_counts().sort_index().to_frame("companies")
dist["pct_of_companies"] = (dist["companies"] / len(per_company) * 100).round(1)
dist["narratives_covered"] = [int(per_company[buckets == b].sum()) for b in dist.index]
dist["pct_of_narratives"] = (dist["narratives_covered"] / per_company.sum() * 100).round(1)
dist

,companies,pct_of_companies,narratives_covered,pct_of_narratives
1,154,43.5,154,4.7
2-4,86,24.3,221,6.8
5-9,38,10.7,244,7.5
10-19,34,9.6,445,13.6
20-49,26,7.3,806,24.6
50+,16,4.5,1402,42.8


## 6. Sanity check — 10 random matched records

In [18]:
pd.set_option("display.max_colwidth", 200)
sample = df[df["has_narrative"]].sample(n=10, random_state=42).copy()
sample["narrative[:200]"] = sample["narrative"].str[:200]
sample[["Complaint ID", "Company", "Sub-product", "Issue", "narrative[:200]"]].reset_index(drop=True)

,Complaint ID,Company,Sub-product,Issue,narrative[:200]
0,14541632,"Upstart Holdings, Inc.",Installment loan,Incorrect information on your report,"In late XXXX, I logged into my account and attempted to update the 'recurring "" autopay payment method to my XXXX XXXX checking account ending in XXXX, which is saved and verified in the system. \..."
1,8564115,SYNCHRONY FINANCIAL,Installment loan,Charged fees or interest you didn't expect,"To Whom It May Concern, On XX/XX/XXXX I purchase a new mattress from a local Mattress Firm : XXXX XXXX XXXX XXXX XXXX XXXX # XXXX XXXX XXXX XXXX XXXX XXXX, CO XXXX Phone : XXXX I was told that the..."
2,11183784,"FirstCash, Inc.",Pawn loan,Can't contact lender or servicer,This company violated my civil rights. And the discriminated against me. \nViolation of U.S. Code S 242 Violation of 18 U.S. Code S 1001 XXXX violated my basic consumer and civil rights by having ...
3,9452341,"Upstart Holdings, Inc.",Installment loan,Struggling to pay your loan,"I have XXXX accounts with upstart. XXXX is 3 months late. Upstart cancelled my auto payments, I haven't been able to access the account as the email address associated with it, I no longer have ac..."
4,11435502,"Paypal Holdings, Inc",Installment loan,Problem when making payments,"Filed billing dispute with this company in a timely manner, the overseas merchant is not helpful in processing the refund. The case ID is : XXXX I have significant proof that the merchandise has b..."
5,8182473,OneMain Finance Corporation,Personal line of credit,Problem with additional add-on products or services,"When I applied for the loan I got it and was told about other things that I would get from having the loan, such as a XXXX dollars gift card every month do to whatever or spend however, I have not..."
6,15445097,JPMORGAN CHASE & CO.,Personal line of credit,Improper use of your report,"XXXX, XXXX JPM made unauthorized inquires over 20 to my credit file and in XXXX."
7,13678692,"CreditNinja Lending, LLC",Payday loan,Charged fees or interest you didn't expect,"I took out a {$1000.00} loan from XXXX XXXX VI, XXXX via XXXX. I was not aware that a {$1400.00} XXXX XXXX XXXX ( CAB ) fee would be added, making the total repayment over {$2500.00} for a {$1000...."
8,7794444,"Affirm Holdings, Inc",Installment loan,Problem when making payments,On XX/XX/XXXX I filed a dispute with Affirm for the loan they issued me and asked them to cancel the loan due to non-receipt of the retail merchandise for which they paid the merchant XXXX XXXX an...
9,8892817,"American First Finance, Inc.",Personal line of credit,Incorrect information on your report,this account was open fraudulently in my name.


In [19]:
# Full text of a couple of them, to confirm these are real narratives.
for cid in sample["Complaint ID"].head(2):
    row = df[df["Complaint ID"] == cid].iloc[0]
    print("=" * 100)
    print(f"Complaint ID {cid}  |  {row['Company']}  |  {row['Sub-product']}")
    print("=" * 100)
    print(row["narrative"][:900])
    print()

Complaint ID 14541632  |  Upstart Holdings, Inc.  |  Installment loan
In late XXXX, I logged into my account and attempted to update the 'recurring " autopay payment method to my XXXX XXXX checking account ending in XXXX, which is saved and verified in the system. 

The website showed the update to the new account went through successfully. However, in the months following, the system attempted withdrawing from the older savings account ending in XXXX, which didnt have sufficient fund. While this happened I updated to the checking account again & made manual payment from the correct checking account ( XXXX ) during this time. 

The company did not take out the autopay payment from the correct account resulting in a payment being late by 30 days. Once I noticed the payment was taken out of the incorrect account, I then immediately paid off the entire loan balance because the website and company was not reliable and did not want to risk further late payment 

Complaint ID 8564115  |  SYN

## Conclusion — is narrative coverage sufficient for lender x sub-product NLP, particularly payday?

In [20]:
pay_all = df[df["Sub-product"] == PAYDAY]
tail = per_company[per_company < 5]
recent = df[df["month"] >= "2026-07"]
print(f"""
OVERALL
  {k:,} of {n:,} complaints ({k / n * 100:.1f}%) have a narrative.
  {df.loc[df.has_narrative, 'Company'].nunique():,} of {df['Company'].nunique():,} companies have at least one.

PAYDAY
  {p_narr:,} of {p_total:,} payday complaints ({p_narr / p_total * 100:.1f}%) have a narrative.
  {len(per_company)} companies have >=1; {int((per_company >= 10).sum())} have >=10; {int((per_company >= 50).sum())} have >=50.
  Median narratives per payday company: {per_company.median():.0f}.

THIN TAIL
  Of the {len(per_company)} payday companies with any narrative, {len(tail)} have fewer than 5,
  and together they hold only {tail.sum() / per_company.sum() * 100:.1f}% of the available payday text.
  The {int((per_company >= 50).sum())} companies with 50+ narratives hold {per_company[per_company >= 50].sum() / per_company.sum() * 100:.1f}% of it.

RECENCY
  Coverage is {df[df.month < '2026-07']['has_narrative'].mean() * 100:.1f}% before 2026-07 and
  {recent['has_narrative'].mean() * 100:.1f}% from 2026-07 onward ({len(recent):,} complaints).
""")


OVERALL
  18,863 of 38,375 complaints (49.2%) have a narrative.
  838 of 1,162 companies have at least one.

PAYDAY
  3,272 of 6,024 payday complaints (54.3%) have a narrative.
  354 companies have >=1; 76 have >=10; 16 have >=50.
  Median narratives per payday company: 2.

THIN TAIL
  Of the 354 payday companies with any narrative, 240 have fewer than 5,
  and together they hold only 11.5% of the available payday text.
  The 16 companies with 50+ narratives hold 42.8% of it.

RECENCY
  Coverage is 54.4% before 2026-07 and
  9.1% from 2026-07 onward (4,477 complaints).



**Verdict: sufficient for a head-of-distribution study, not for a full-coverage model, and
not for tracking recent behaviour.**

What the numbers support:

- **Payday coverage is real and usable at the top.** 54.3% of payday complaints carry a
  narrative, and 42 payday lenders have 20+ narratives, 16 have 50+. That is enough text
  to characterise the largest payday lenders, and the top 16 companies hold a
  disproportionate share of the available text.
- **Coverage is fairly even across `Sub-product`** (46-58%), so payday is not disadvantaged
  relative to installment or title. There is no sub-product-specific coverage problem to solve.

What the numbers rule out:

- **The long tail is unmodellable.** The median payday company has 2 narratives. Any
  per-lender model would either cover a few dozen lenders or produce statistically
  meaningless estimates for the other 300+. This is the same sparsity problem the
  structured fields have, only milder.
- **Nothing recent can be analysed.** The archive's narrative supply goes to zero from
  2026-08 onward (Aug and Sep 2026 are both 0.0%, and the archive itself only claims
  coverage through 2026-08-14). Archive-wide narrative supply fell from ~26% in mid-2025
  to 3.3% in Jan-Feb 2026, 1.5% in Jul 2026, and 1 record in all of Aug 2026. Any NLP
  finding would describe 2023-2025 behaviour and could not be refreshed.
- **The text is a biased sample even where it exists.** Narratives are self-selected:
  consumers who chose to write one. There is no way to test whether narrative-having
  complaints differ systematically from the rest.

Practical consequence: NLP here can support a *qualitative, lender-level* read on roughly
40-70 payday lenders over 2023-2025. It cannot support a scored ranking across the full
lender set, and it cannot support any claim about current conditions. Pairing it with the
structured fields for the head of the distribution is the only defensible use.

Caveats on this measurement:

- The archive is a set of FOIA production snapshots, not the live database, so 1,053 of
  our complaints (2.7%, almost all from 2026-09) are absent for reasons of archive
  coverage rather than absence of text.
- Our extract is a **stale snapshot**: 284 matched complaints carry a different company
  name in the archive (254 `Ningo Lending LLC dba Spotloan` -> `BlueChip Financial`, 29
  `STATE EMPLOYEES'` -> `STATE EMPLOYEES`, 1 `American First Funding` ->
  `LAKEVIEW LENDING`). Acquisitions and renames will need entity resolution before any
  lender-level text work, and the `Company` values in our CSV are already out of date.